In [1]:
import numpy as np
import pandas as pd

In [2]:
import requests

url = "https://www.gutenberg.org/files/11/11-0.txt"
text = requests.get(url).text

print(text[:1000])  # preview

*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence




CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

So she was considering in her

In [3]:
start = text.find("CHAPTER I")
end = text.find("*** END")

text = text[start:end]

In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [5]:
import re
text=text.lower()
text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation
text = re.sub(r'\s+', ' ', text)         # remove extra spaces

In [6]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])
word_index=tokenizer.word_index
vocab_size=len(word_index)+1;
print(vocab_size)

2752


In [14]:
input_sequences = []

for line in text.split('.'):
    tokens = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(tokens)):
        n_gram = tokens[:i+1]
        input_sequences.append(n_gram)

In [26]:
max_len=15
input_sequences = [
    seq[-max_len:] for seq in input_sequences
]
input_sequences=pad_sequences(input_sequences,maxlen=max_len,padding='pre')

In [27]:
input_sequences

array([[   0,    0,    0, ...,    0,  174,    9],
       [   0,    0,    0, ...,  174,    9,   35],
       [   0,    0,    0, ...,    9,   35,    1],
       ...,
       [   0,    0,    0, ..., 2751, 1461,  796],
       [   0,    0,    0, ..., 1461,  796,    1],
       [   0,    0,    0, ...,  796,    1,  213]], dtype=int32)

In [28]:
X=input_sequences[:,:-1]
y=input_sequences[:,-1]

In [29]:
X.shape

(26450, 14)

In [30]:
y.shape

(26450,)

In [31]:
y=tf.keras.utils.to_categorical(y,num_classes=vocab_size)

In [32]:
from tensorflow.keras.layers import Input
model = Sequential([
    Input(shape=(max_len-1,)),
    Embedding(vocab_size, 64),
    LSTM(128,recurrent_activation='sigmoid'),
    Dense(vocab_size, activation='softmax')
])

In [33]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 14, 64)         │       176,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2752)           │       355,008 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 629,952 (2.40 MB)

 Trainable params: 629,952 (2.40 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
model.fit(X, y, epochs=20, batch_size=64)

Epoch 1/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.0615 - loss: 6.2485
Epoch 2/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.0670 - loss: 5.8939
Epoch 3/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.0799 - loss: 5.6864
Epoch 4/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.0982 - loss: 5.4749
Epoch 5/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1112 - loss: 5.2877
Epoch 6/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1226 - loss: 5.1232
Epoch 7/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1310 - loss: 4.9774
Epoch 8/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1398 - loss: 4.8449
Epoch 9/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.1476 - loss: 4.7195
Epoch 10/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1583 - loss: 4.6024
Epoch 11/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1680 - loss: 4.4856
Epoch 12/20
414/414 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step

In [46]:
def apply_temperature(probs, temperature):
    probs = np.asarray(probs).astype('float64')
    probs = np.log(probs + 1e-10) / temperature
    probs = np.exp(probs)
    return probs / np.sum(probs)

def predict_next_word(seed_text, next_words=5, temperature=1.0, top_k=5):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([seed_text])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')

        probs = model.predict(seq, verbose=0)[0]

        probs = apply_temperature(probs, temperature)

        top_k_indices = np.argsort(probs)[-top_k:]
        top_k_probs = probs[top_k_indices]
        top_k_probs = top_k_probs / np.sum(top_k_probs)

        next_index = np.random.choice(top_k_indices, p=top_k_probs)
        next_word = tokenizer.index_word.get(next_index, "")

        seed_text += " " + next_word

    return seed_text

In [49]:
predict_next_word("alice friends", next_words=5, temperature=0.8, top_k=5)

'alice friends said the hatter who is'